# Anomaly Detection in Credit Card Transaction Data

This notebook treats fraud labels as unavailable during training and uses them only for evaluation. The goal is not to maximize a leaderboard metric; it is to design a practical anomaly scoring system that reduces analyst review burden while failing safely for a high-risk financial workflow.

## 1. Exploration and framing

Operational framing:

- The current rule system flags about 5% of transactions.
- Analysts estimate roughly 90% of flagged transactions are false positives.
- Missing fraud is more dangerous than reviewing extra false positives.
- I therefore choose a default operating point based on review capacity: flag the top 5% highest-risk transactions and evaluate whether the model captures more fraud at the same review burden.

I also evaluate lower and higher review rates to show the precision/recall tradeoff.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_scorer import FEATURE_COLUMNS, FraudAnomalyScorer

DATA_PATH = PROJECT_ROOT / "data" / "creditcard.csv"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "fraud_count": int(df["Class"].sum()),
    "fraud_rate": float(df["Class"].mean()),
    "missing_values": int(df.isna().sum().sum()),
}
summary

## 2. Approach and implementation

Primary approach: Isolation Forest.

Why this model first:

- It is genuinely unsupervised, matching the training constraint.
- It scales better than distance-to-all-points approaches.
- It returns a continuous anomaly score that can be thresholded by analyst capacity.
- It is simple enough to explain to an engineering manager or compliance stakeholder.

The production scoring logic lives in `src/fraud_scorer.py`, which bundles preprocessing, model, threshold, feature contract, version, validation, and serialization.

In [ ]:
# Preserve temporal order for evaluation. Labels are not used for fitting.
train_fraction = 0.70
split_idx = int(len(df) * train_fraction)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

scorer = FraudAnomalyScorer.fit(
    train_df,
    review_rate=0.05,
    random_state=RANDOM_STATE,
    n_estimators=200,
)
scores = scorer.score_batch(test_df)
eval_df = test_df[["Class"]].join(scores)
eval_df.head()

## 3. Evaluation at operating points

Accuracy is not useful here because the non-fraud class dominates. Instead, I evaluate at fixed review rates:

- How many transactions are sent to analysts?
- How many fraud cases are captured?
- What is precision among reviewed cases?
- How many false positives occur per true positive?

This maps model performance directly to analyst workload.

In [ ]:
def evaluate_at_review_rate(frame: pd.DataFrame, review_rate: float) -> dict:
    threshold = frame["anomaly_score"].quantile(1 - review_rate)
    flagged = frame["anomaly_score"] >= threshold
    true_fraud = frame["Class"] == 1
    true_positives = int((flagged & true_fraud).sum())
    false_positives = int((flagged & ~true_fraud).sum())
    false_negatives = int((~flagged & true_fraud).sum())
    reviewed = int(flagged.sum())
    return {
        "review_rate": review_rate,
        "reviewed_transactions": reviewed,
        "captured_fraud": true_positives,
        "missed_fraud": false_negatives,
        "precision": true_positives / reviewed if reviewed else 0,
        "recall": true_positives / int(true_fraud.sum()) if true_fraud.sum() else 0,
        "false_positives_per_true_positive": false_positives / true_positives if true_positives else np.inf,
    }

pd.DataFrame([evaluate_at_review_rate(eval_df, rate) for rate in [0.01, 0.02, 0.05, 0.10]])

## 4. Productionization

The production interface is implemented in `FraudAnomalyScorer`.

It provides:

- `score_one(transaction)` for individual scoring;
- `score_batch(df)` for batch or streaming micro-batch scoring;
- strict input validation for missing columns, non-numeric data, NaN, and infinite values;
- bundled preprocessing + model + threshold + version;
- save/load behavior through `joblib`.

Tests live in `tests/test_fraud_scorer.py` and include serialization round-trip and bad-input rejection.

In [ ]:
artifact_path = PROJECT_ROOT / "artifacts" / "fraud_scorer.joblib"
scorer.save(artifact_path)
loaded = FraudAnomalyScorer.load(artifact_path)

single_result = loaded.score_one(test_df.iloc[0][list(FEATURE_COLUMNS)])
single_result

## 5. Operating the system

Monitoring signals I would track daily and per client:

- transaction volume by client and source system;
- missing/invalid feature rates;
- `Amount` distribution drift;
- anomaly-score distribution drift;
- flagged review rate vs configured review budget;
- analyst-confirmed fraud rate once delayed labels arrive;
- override/escalation rate from analysts.

Safe failure modes:

- If required fields are missing or invalid, reject scoring loudly and route to fallback/manual review.
- If the model service is unavailable, fall back to existing conservative rules.
- If score distributions shift sharply, freeze promotion of new thresholds and alert before automatic retraining.

Retraining:

- Train candidate models on recent validated data.
- Evaluate with delayed labels where available and with proxy drift metrics where labels are incomplete.
- Promote only if candidate improves fraud capture at fixed review budget without unacceptable drift or instability.
- Keep previous model artifacts for rollback.

## 6. Scale and multi-tenancy

I would start with a hybrid design:

- a shared global model for clients with little history;
- per-client thresholds based on volume, staffing, and risk tolerance;
- per-client or segment-specific models once enough clean history exists.

A brand-new client starts with the global model and a conservative threshold. One client's bad data should be isolated through per-client data quality monitoring, tenant-level feature validation, and model training datasets that can exclude or down-weight unhealthy tenants.

## 7. Recommendation

First production path: ship a conservative Isolation Forest anomaly scorer behind the existing rule-based workflow. Use it to prioritize analyst queues at the current 5% review rate, not to automatically decline or clear transactions. This gives the organization a measurable improvement path while limiting risk.